# Step 1: Knowledge Graph Approach for R(u,c)

Pipeline: user → video → ccid → concept → course

**Formulas:**
```
watch(u,c) = 0.7 × (|u∩c|/|c|) + 0.3 × (|u∩c|/|u|)  (Hybrid relevance)
R(u,c) = 0.3·enroll + 0.7·watch
```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("Step1_KnowledgeGraph") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .getOrCreate()

print("Step 1: Knowledge Graph Approach")

Step 1: Knowledge Graph Approach


In [2]:
# Configuration
W_ENROLL = 0.3
W_WATCH = 0.7
MIN_COURSES_USER = 2    
MIN_USERS_COURSE = 3   


print(f"Updated thresholds: user_min={MIN_COURSES_USER}, course_min={MIN_USERS_COURSE}")

DATA_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data"
OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"WEIGHTS (from Proposal):")
print(f"  enroll = {W_ENROLL}")
print(f"  watch = {W_WATCH}")
print(f"  Sum = {W_ENROLL + W_WATCH }")
print(f"\nFilter: user_min_courses={MIN_COURSES_USER}, course_min_users={MIN_USERS_COURSE}")

Updated thresholds: user_min=2, course_min=3
WEIGHTS (from Proposal):
  enroll = 0.3
  watch = 0.7
  Sum = 1.0

Filter: user_min_courses=2, course_min_users=3


## Load data Knowledge Graph Mappings

In [3]:
# 1. video_id → ccid
video_ccid = spark.read.csv(
    f"{DATA_PATH}/relations/video_id-ccid.txt",
    sep="\t", header=False, inferSchema=True
).toDF("video_id", "ccid")

# 2. ccid → concept
ccid_concept = spark.read.csv(
    f"{DATA_PATH}/relations/concept-video.txt",
    sep="\t", header=False, inferSchema=True
).toDF("concept", "ccid")

# 3. problem_id → concept
problem_concept = spark.read.csv(
    f"{DATA_PATH}/relations/concept-problem.txt",
    sep="\t", header=False, inferSchema=True
).toDF("concept", "problem_id")

# 4. concept → course
concept_course = spark.read.csv(
    f"{DATA_PATH}/relations/concept-course.csv", header=True
)
# Load user-video
user_video_df = spark.read.json(f"{DATA_PATH}/relations/user-video.json")
#Load user
user_df = spark.read.json(f"{DATA_PATH}/entities/user.json")
#Load course 
course_df = spark.read.json(f"{DATA_PATH}/entities/course.json")

print(f"video-ccid: {video_ccid.count():,}")
print(f"ccid-concept: {ccid_concept.count():,}")
print(f"problem-concept: {problem_concept.count():,}")
print(f"concept-course: {concept_course.count():,}")
print(f"course: {course_df.count():,}")
print(f"user-video: {user_video_df.count():,}")
print(f"user: {user_df.count():,}")

video-ccid: 2,798,892
ccid-concept: 624,683
problem-concept: 33,180
concept-course: 451,078
course: 3,781
user-video: 299,938
user: 498,916


## Build Course → Concepts Mapping

In [4]:
# Course concepts from concept-course.csv directly
course_concepts = concept_course.select(
      col("course_id"),
      col("term").alias("concept")  # Đổi tên cột
  ).distinct()

# Count concepts per course
course_concept_count = course_concepts.groupBy("course_id").count()\
    .withColumnRenamed("count", "total_concepts")

print(f"Courses with concepts: {course_concepts.select('course_id').distinct().count():,}")
print(f"Total concept-course pairs: {course_concepts.count():,}")
course_concept_count.show(5)

Courses with concepts: 887
Total concept-course pairs: 451,078
+---------+--------------+
|course_id|total_concepts|
+---------+--------------+
| C_883345|          1484|
| C_682225|           215|
| C_676937|           932|
| C_680860|           237|
| C_697064|           581|
+---------+--------------+
only showing top 5 rows


## Signal 1: Enrollment

In [5]:
enroll_df = user_df.select(
    col("id").alias("user_id"),
    posexplode("course_order").alias("pos", "course_id"),
    col("enroll_time")
).withColumn("enroll_time_str", element_at(col("enroll_time"), col("pos") + 1))\
 .withColumn("enroll_signal", lit(1.0))\
 .select(
     "user_id",
     concat(lit("C_"), col("course_id").cast("string")).alias("course_id"),
     "enroll_time_str",
     "enroll_signal"
 )

print(f"Enrollment pairs: {enroll_df.count():,}")
enroll_df.show()

Enrollment pairs: 1,163,400
+-------+---------+-------------------+-------------+
|user_id|course_id|    enroll_time_str|enroll_signal|
+-------+---------+-------------------+-------------+
|   U_22| C_682129|2019-10-12 10:28:02|          1.0|
|   U_22|C_2294668|2020-11-21 14:03:28|          1.0|
|   U_24| C_597214|2019-05-20 16:06:48|          1.0|
|   U_24| C_605512|2019-05-24 19:34:43|          1.0|
|   U_24| C_597211|2019-06-11 02:50:04|          1.0|
|   U_24| C_597314|2019-06-12 17:22:07|          1.0|
|   U_24| C_597208|2019-06-17 15:22:41|          1.0|
|   U_24| C_629501|2019-08-15 16:03:51|          1.0|
|   U_24| C_597159|2019-08-15 16:18:57|          1.0|
|   U_24| C_605994|2019-08-15 16:26:24|          1.0|
|   U_24| C_738188|2019-09-11 18:39:36|          1.0|
|   U_24| C_682129|2019-10-06 16:23:51|          1.0|
|   U_24| C_674910|2019-10-08 17:51:04|          1.0|
|   U_24| C_817295|2019-10-09 14:16:43|          1.0|
|   U_24| C_813711|2019-10-11 17:38:33|          1.0|


## Signal 2: Watch Score (via Knowledge Graph)

Path: user → video → ccid → concept → course

**Formula (Hybrid):**
```
course_relevance = |user_concepts ∩ course_concepts| / |course_concepts|
user_relevance = |user_concepts ∩ course_concepts| / |user_concepts|
watch_signal = 0.7 × course_relevance + 0.3 × user_relevance
```

**Rationale:**
- 70% course-relevance: Measures preparedness for course (main objective)
- 30% user-relevance: Rewards learners who explore broadly (not penalized)

In [ ]:
# Extract video_ids watched by each user
user_videos = user_video_df.select(
    "user_id",
    explode("seq").alias("item")
).select("user_id", col("item.video_id").alias("video_id"))\
.distinct()

print(f"User-video interactions: {user_videos.count():,}")

# user → video → ccid → concept
user_concepts_watch = user_videos.join(
    video_ccid, "video_id", "inner"
).join(
    ccid_concept, "ccid", "inner"
).select("user_id", "concept").distinct()

print(f"User-concept (from videos): {user_concepts_watch.count():,}")

# Calculate WATCH SIGNAL: Hybrid of Course-relevance (70%) + User-relevance (30%)

# 1. Count matched concepts per (user, course)
user_course_concept_match = user_concepts_watch.join(
    course_concepts, "concept", "inner"
).groupBy("user_id", "course_id").count()\
.withColumnRenamed("count", "matched_concepts")

# 2. Get total concepts per course (|c|) - from course_concept_count
# 3. Calculate course_relevance: |u ∩ c| / |c|
user_course_course_rel = user_course_concept_match.join(
    course_concept_count, "course_id"
).withColumn(
    "course_relevance",
    col("matched_concepts") / col("total_concepts")
)

# 4. Get total concepts per user (|u|)
user_concept_count = user_concepts_watch.groupBy("user_id").count() \
    .withColumnRenamed("count", "total_user_concepts")

# 5. Calculate user_relevance: |u ∩ c| / |u|
user_course_hybrid = user_course_course_rel.join(
    user_concept_count, "user_id"
).withColumn(
    "user_relevance",
    col("matched_concepts") / col("total_user_concepts")
)

# 6. Hybrid: 0.7 * course_relevance + 0.3 * user_relevance
user_course_watch = user_course_hybrid.withColumn(
    "watch_signal",
    0.7 * col("course_relevance") + 0.3 * col("user_relevance")
).select("user_id", "course_id", "watch_signal", "course_relevance", "user_relevance")

print(f"Watch pairs: {user_course_watch.count():,}")
user_course_watch.show()

User-video interactions: 21,053,655
User-concept (from videos): 160,020,500
Watch pairs: 6,366,938


Combine signal

In [7]:
# Start with enrollment
R_matrix = enroll_df.select("user_id", "course_id", "enroll_time_str", "enroll_signal")

# Join watch signal 
R_matrix = R_matrix.join(
    user_course_watch.select("user_id", "course_id", "watch_signal"),
    ["user_id", "course_id"],
    "inner"
).withColumn("watch_signal", coalesce(col("watch_signal"), lit(0.0)))


# Calculate R_score
R_matrix = R_matrix.withColumn(
    "R_score",
    W_ENROLL * col("enroll_signal") + W_WATCH * col("watch_signal")
)

print(f"R(u,c) before filter: {R_matrix.count():,}")
R_matrix.show(10)

R(u,c) before filter: 26,853
+----------+---------+-------------------+-------------+-------------------+-------------------+
|   user_id|course_id|    enroll_time_str|enroll_signal|       watch_signal|            R_score|
+----------+---------+-------------------+-------------+-------------------+-------------------+
| U_1004659| C_735164|2020-09-15 13:42:15|          1.0| 0.3071428571428571| 0.5149999999999999|
|U_10066720| C_676932|2020-02-03 21:43:00|          1.0|0.35121951219512193| 0.5458536585365853|
|U_10075048| C_676932|2020-05-19 13:38:46|          1.0|0.40243902439024387| 0.5817073170731707|
| U_1008050| C_735164|2020-09-16 13:51:53|          1.0|0.40714285714285714|              0.585|
| U_1008778| C_676932|2020-03-25 11:41:54|          1.0|0.35121951219512193| 0.5458536585365853|
| U_1009481| C_735164|2020-09-20 09:13:46|          1.0|0.32857142857142857|               0.53|
|U_10099723| C_697791|2020-03-03 18:03:03|          1.0|0.34098250336473757| 0.5386877523553163|
|

## Filter & Normalize

In [8]:
# Filter
valid_users = R_matrix.groupBy("user_id").count().filter(col("count") >= MIN_COURSES_USER)
valid_courses = R_matrix.groupBy("course_id").count().filter(col("count") >= MIN_USERS_COURSE)

R_filtered = R_matrix.join(valid_users.select("user_id"), "user_id")\
                     .join(valid_courses.select("course_id"), "course_id")


print(f"Valid users (>= {MIN_COURSES_USER} course): {valid_users.count():,}")
print(f"Valid courses (>= {MIN_USERS_COURSE} user): {valid_courses.count():,}")
print(f"\nFinal R(u,c) matrix: {R_filtered.count():,}")

# Normalize per user
user_window = Window.partitionBy("user_id")
R_filtered = R_filtered.withColumn("R_max", max("R_score").over(user_window))\
                       .withColumn("R_min", min("R_score").over(user_window))

R_final = R_filtered.withColumn(
    "R_norm",
    when(col("R_max") > col("R_min"),
         (col("R_score") - col("R_min")) / (col("R_max") - col("R_min")))
    .when(col("R_max") > 0, 1.0)
    .otherwise(0.0)
).drop("R_max", "R_min")

print(f"Final R(u,c): {R_final.count():,}")
print(f"Unique users: {R_final.select('user_id').distinct().count():,}")
print(f"Unique courses: {R_final.select('course_id').distinct().count():,}")

Valid users (>= 2 course): 5,039
Valid courses (>= 3 user): 475

Final R(u,c) matrix: 18,370
Final R(u,c): 18,370
Unique users: 5,033
Unique courses: 470


## Statistics

In [9]:
print("R_score statistics:")
R_final.select("R_score").summary().show()

print("\nSignal composition:")
R_final.select(
    avg("enroll_signal").alias("avg_enroll"),
    avg("watch_signal").alias("avg_watch"),
    avg("R_score").alias("avg_R_score")
).show()

print("\nNon-zero signals:")
R_final.select(
    (sum(when(col("watch_signal") > 0, 1)) / count("*")).alias("pct_watch"),
).show()

R_score statistics:
+-------+-------------------+
|summary|            R_score|
+-------+-------------------+
|  count|              18370|
|   mean| 0.4546915527768241|
| stddev|0.10292140495385879|
|    min|  0.300679919627207|
|    25%|0.38760738655341886|
|    50%|0.42517381974248925|
|    75%| 0.5260130718954248|
|    max|                1.0|
+-------+-------------------+


Signal composition:
+----------+-------------------+-----------------+
|avg_enroll|          avg_watch|      avg_R_score|
+----------+-------------------+-----------------+
|       1.0|0.22098793253832494|0.454691552776827|
+----------+-------------------+-----------------+


Non-zero signals:
+---------+
|pct_watch|
+---------+
|      1.0|
+---------+



## Save Output

In [10]:
#R_final.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/R_matrix_kg")
R_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{OUTPUT_PATH}/R_matrix")

print("Saved!")

Saved!


In [11]:
user_course_watch.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{OUTPUT_PATH}/user_course_watch")